# ESMC Local - Protein Language Model Embeddings

Run ESMC locally to generate protein sequence embeddings and compute mutation scores.

**Key Features:**
- Load ESMC model locally (600M or 300M variants)
- Extract embeddings from protein sequences
- Compute Shannon entropy for conservation analysis
- Score mutations using log-likelihood ratios
- Works with GPU acceleration (Colab A100/T4)


In [ ]:
!pip install -q torch transformers accelerate einops biotite biopython scikit-learn pandas tqdm ipywidgets

In [ ]:
# Install ESM from GitHub
!pip install -q git+https://github.com/Biohub/esm.git@main

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Tuple
import json
from tqdm import tqdm
from IPython.display import display, Markdown

# ESM imports
from esm.models import ESMCForCausalLM
from esm.tokenization import TokenizerBase
from huggingface_hub import hf_hub_download

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Configuration

In [ ]:
# Model selection
MODEL_NAME = "esmc_600m"  # Options: "esmc_600m", "esmc_300m", "esmc_300m_open_v1"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 8

# Data
OUTPUT_DIR = "/content/esmc_output"  # Change for local runs
Path(OUTPUT_DIR).mkdir(exist_ok=True)

print(f"Model: {MODEL_NAME}")
print(f"Device: {DEVICE}")
print(f"Output directory: {OUTPUT_DIR}")

## Load ESMC Model

In [ ]:
print(f"Loading {MODEL_NAME}...")
model = ESMCForCausalLM.from_pretrained(f"esm2/esmc_{MODEL_NAME}").to(DEVICE)
model.eval()
tokenizer = TokenizerBase()
print(f"Model loaded successfully. Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

## Utility Functions

In [ ]:
def load_fasta(filepath: str) -> Dict[str, str]:
    """Load FASTA file into dictionary."""
    sequences = {}
    with open(filepath) as f:
        current_id = None
        current_seq = []
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if current_id is not None:
                    sequences[current_id] = "".join(current_seq)
                current_id = line[1:].split()[0]
                current_seq = []
            else:
                current_seq.append(line)
        if current_id is not None:
            sequences[current_id] = "".join(current_seq)
    return sequences

def compute_entropy(logits: np.ndarray) -> np.ndarray:
    """Compute Shannon entropy from logits."""
    probs = F.softmax(torch.from_numpy(logits), dim=-1).numpy()
    entropy = -np.sum(probs * np.log(probs + 1e-10), axis=-1)
    return entropy

def score_mutation(ref_logits: np.ndarray, mut_logits: np.ndarray, ref_aa: str, mut_aa: str) -> float:
    """Score mutation using log-likelihood ratio."""
    ref_prob = F.softmax(torch.from_numpy(ref_logits), dim=-1)
    mut_prob = F.softmax(torch.from_numpy(mut_logits), dim=-1)
    
    # Simple mutation scoring - log ratio of probabilities
    score = float(torch.log(mut_prob.max()) - torch.log(ref_prob.max()))
    return score

## Generate Embeddings

In [ ]:
#@markdown Upload FASTA file or paste sequence

from google.colab import files
import io

input_mode = "paste"  # "upload" or "paste"

if input_mode == "upload":
    print("Upload your FASTA file:")
    uploaded = files.upload()
    fasta_file = list(uploaded.keys())[0]
    sequences = load_fasta(fasta_file)
else:
    # Example sequences
    sequences = {
        "example_1": "MKVLIVAALLLAVGLAFWECEKRKYQCPEKPQE",
        "example_2": "MDVFMGVGVVDAKALVDYLVPGQDTAV"
    }

print(f"Loaded {len(sequences)} sequences")
for seq_id, seq in list(sequences.items())[:3]:
    print(f"  {seq_id}: {len(seq)} aa")

In [ ]:
embeddings = {}
entropies = {}

with torch.no_grad():
    for seq_id, sequence in tqdm(sequences.items()):
        # Tokenize
        tokens = tokenizer.tokenize(sequence)
        tokens = torch.tensor([tokens]).to(DEVICE)
        
        # Forward pass
        output = model(tokens, output_hidden_states=True)
        hidden_states = output.hidden_states[-1]  # Last layer
        
        # Extract embeddings (remove special tokens)
        embedding = hidden_states[0, 1:-1, :].cpu().numpy()  # [seq_len, hidden_size]
        embeddings[seq_id] = embedding
        
        # Compute entropy from logits
        logits = output.logits[0, 1:-1, :].cpu().numpy()
        entropy = compute_entropy(logits)
        entropies[seq_id] = entropy

print(f"\nProcessed {len(embeddings)} sequences")
for seq_id in list(embeddings.keys())[:1]:
    print(f"  {seq_id}: embedding shape {embeddings[seq_id].shape}")

## Analysis & Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(entropies), 1, figsize=(14, 3*len(entropies)))
if len(entropies) == 1:
    axes = [axes]

for idx, (seq_id, entropy) in enumerate(entropies.items()):
    ax = axes[idx]
    ax.plot(entropy, linewidth=2)
    ax.fill_between(range(len(entropy)), entropy, alpha=0.3)
    ax.set_title(f"{seq_id} - Shannon Entropy (Conservation Analysis)")
    ax.set_xlabel("Position")
    ax.set_ylabel("Entropy")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/entropy_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

display(Markdown("**High entropy = flexible/variable positions | Low entropy = conserved/constrained positions"))

## Save Results

In [ ]:
# Save embeddings as numpy arrays
for seq_id, embedding in embeddings.items():
    np.save(f"{OUTPUT_DIR}/{seq_id}_embedding.npy", embedding)

# Save entropy analysis as CSV
for seq_id, entropy in entropies.items():
    df = pd.DataFrame({
        "position": range(len(entropy)),
        "entropy": entropy
    })
    df.to_csv(f"{OUTPUT_DIR}/{seq_id}_entropy.csv", index=False)

print(f"Results saved to {OUTPUT_DIR}")
print(f"  - Embeddings: {{seq_id}}_embedding.npy")
print(f"  - Entropy analysis: {{seq_id}}_entropy.csv")